# Used Car Price Prediction - Exploratory Data Analysis (EDA)

This notebook conducts exploratory data analysis on the CarDekho dataset.
We investigate dataset distributions, missing values, duplicates, outliers, and feature-target relationships.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set styling
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 5)

data_path = Path("../data/raw/cardekho_dataset.csv")
if not data_path.exists():
    data_path = Path("data/raw/cardekho_dataset.csv")

df = pd.read_csv(data_path)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Dataset Shape:", df.shape)
df.head()

## 1. Summary Statistics & Data Quality
Check columns, data types, missing values, and unique counts.

In [ ]:
info_df = pd.DataFrame({
    "Dtype": df.dtypes,
    "Missing Values": df.isnull().sum(),
    "Missing (%)": (df.isnull().sum() / len(df) * 100).round(2),
    "Unique Count": df.nunique()
})
print(info_df)
print("\nDuplicate rows (excluding index):", df.duplicated().sum())

## 2. Target Variable: Selling Price Distribution
The target price in INR is heavily right-skewed. Taking `log1p(price)` normalizes the target distribution for regression.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df["selling_price"], bins=50, kde=True, ax=axes[0], color="royalblue")
axes[0].set_title("Original Selling Price Distribution (INR)")
axes[0].set_xlabel("Selling Price (INR)")

sns.histplot(np.log1p(df["selling_price"]), bins=50, kde=True, ax=axes[1], color="crimson")
axes[1].set_title("Log-Transformed Selling Price (log1p)")
axes[1].set_xlabel("log1p(Selling Price)")

plt.tight_layout()
plt.show()

## 3. Depreciation Dynamics: Vehicle Age and Odometer KM
As vehicles age and accumulate mileage, market value exhibits steady depreciation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.lineplot(data=df, x="vehicle_age", y="selling_price", ax=axes[0], color="darkorange", marker="o")
axes[0].set_title("Vehicle Age vs Mean Selling Price")
axes[0].set_xlabel("Vehicle Age (Years)")
axes[0].set_ylabel("Selling Price (INR)")

sns.scatterplot(data=df[df["km_driven"] < 300000], x="km_driven", y="selling_price", alpha=0.3, ax=axes[1], color="forestgreen")
axes[1].set_title("KM Driven vs Selling Price")
axes[1].set_xlabel("Kilometers Driven")
axes[1].set_ylabel("Selling Price (INR)")

plt.tight_layout()
plt.show()

## 4. Transmission, Fuel Type, and Brand Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="transmission_type", y="selling_price", ax=axes[0], palette="Set2")
axes[0].set_title("Selling Price by Transmission Type")
axes[0].set_ylim(0, 4000000)

top_brands = df["brand"].value_counts().head(10).index
sns.barplot(data=df[df["brand"].isin(top_brands)], x="brand", y="selling_price", ax=axes[1], palette="tab10", errorbar=None)
axes[1].set_title("Average Selling Price for Top 10 Brands")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 5. Numeric Correlation Matrix
Engine capacity and max power demonstrate high positive correlation with vehicle valuation.

In [ ]:
num_cols = ["vehicle_age", "km_driven", "mileage", "engine", "max_power", "seats", "selling_price"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title("Numeric Features Correlation Heatmap")
plt.show()

## 6. Key Conclusions for the ML Pipeline
- **Target Transformation**: `TransformedTargetRegressor` with `np.log1p` and `np.expm1` is mandatory to handle right-skewed car prices.
- **Outlier Strategy**: Removing data corruptions (identical km and price) and clipping extreme outliers (>350k km, >20 yrs age) prevents distortion.
- **Category Grouping**: Rare brands and models (<10 occurrences) are mapped to `Other` to avoid overfitting on unrepresentative samples.
- **Model Selection**: Random Forest Regressor captures non-linear interactions (e.g. premium brands maintain higher baseline value despite age) achieving ~0.92 R².